# CDC downloads — getting the raw files back

The two file-delivered CDC sources have no API to re-query on demand, so the
raw files are fetched once and parsed from disk. This notebook is how they are
fetched, and its saved output is the record that they were.

`notebooks/cdc/data/` is gitignored: 22M of workbooks, regenerable by running
the cells below. What is *not* regenerable is knowing how — hence this
notebook rather than a script.

In [1]:
import sys
sys.path.append("../")     # notebooks/<source>/ helpers
sys.path.append("../../")  # repo root

import fetch
import nvsr_series as ns

## NVSR — 936 Excel workbooks off FTP

Everything here is idempotent: a workbook already on disk is skipped, so
re-running resumes rather than re-downloads. That matters more than usual —
`ftp.cdc.gov`'s bot filter is **rate-based**, and a pull at ~3 files/s served
400 workbooks before stalling every subsequent read to a timeout. `fetch` uses
`curl_cffi` with a browser fingerprint and a 1s pause; don't lower it.

Two naming quirks the fetcher absorbs, so the local tree is uniform whatever
the remote year called things:

| | quirk |
| --- | --- |
| 2018 state (`70-01`) | files are `Alabama-1-Total.xlsx`, not `AL1.xlsx` |
| 2020 national (`71-01`) | lowercase `table01.xlsx` |
| every state year | `{ST}4` is **standard errors**, not a life table — skipped |

Expect ~15 minutes on a cold run, seconds when everything is present.

In [2]:
national = fetch.fetch_national()

  2018 (69-12): {'wanted': 12, 'downloaded': 0, 'on_disk': 12}


  2019 (70-19): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}


  2020 (71-01): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}


  2021 (72-12): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}


  2022 (74-02): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}
  2023 (74-06): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}


  2024 (75-05): {'wanted': 18, 'downloaded': 0, 'on_disk': 18}


In [3]:
state = fetch.fetch_state()

  2018 (70-01): {'wanted': 153, 'downloaded': 0, 'on_disk': 153}


  2019 (70-18): {'wanted': 153, 'downloaded': 0, 'on_disk': 153}


  2020 (71-02): {'wanted': 153, 'downloaded': 0, 'on_disk': 153}


  2021 (73-07): {'wanted': 153, 'downloaded': 0, 'on_disk': 153}


  2022 (74-12): {'wanted': 153, 'downloaded': 0, 'on_disk': 204}


## Check the parse before trusting the files

A truncated or substituted workbook still parses — it just returns a wrong
number. Both guards run over everything on disk and raise rather than
returning quietly-wrong values.

`verify_national` compares age-0 life expectancy against the published
figures. `verify_state` has no published table to check against (51
jurisdictions a year is too many to transcribe), so it leans on three things
a misread cannot satisfy: `female > both > male`, `e0` within 60–95, and the
national value falling inside the state range.

In [4]:
print("national e0 vs published:")
for year, value in sorted(ns.verify_national().items()):
    print(f"  {year}  {value:.4f}  -> {round(value, 1)}  (published {ns.PUBLISHED_E0[year]})")

print("\nstate jurisdictions per year:", ns.verify_state())

national e0 vs published:
  2018  78.7375  -> 78.7  (published 78.7)
  2019  78.8482  -> 78.8  (published 78.8)
  2020  76.9950  -> 77.0  (published 77.0)
  2021  76.3702  -> 76.4  (published 76.4)
  2022  77.4569  -> 77.5  (published 77.5)
  2023  78.4214  -> 78.4  (published 78.4)
  2024  78.9710  -> 79.0  (published 79.0)



state jurisdictions per year: {2018: 51, 2019: 51, 2020: 51, 2021: 51, 2022: 51}


## WONDER — 9 concepts, throttled

**Not reproducible from this repo today.** The cached JSON under `data/wonder/`
holds results (`year, deaths, population, crude_rate, age_adjusted_rate`), not
the queries that produced them, and the ICD-10 code sets for 8 of the 9
concepts are not recorded anywhere — only `ALCOHOL_INDUCED_ICD10` survives, in
`wonder.ipynb`. `_download_summary.json` keeps the citation and a code *count*
per concept, which is enough to check a reconstruction against but not to
rebuild one from.

Reconstructing them means re-deriving each set from its NVSR citation and
verifying against the cached deaths counts year by year — one throttled query
per concept, so ~20 minutes, and worth doing deliberately rather than inline
here. Until then, treat `data/wonder/` as **irreplaceable**: the normalized
`data/timeseries/cdc_wonder.jsonl` is tracked in git for that reason.

In [5]:
import json
from pathlib import Path

summary = json.loads(Path("data/wonder/_download_summary.json").read_text())
print(f"{len(summary)} concepts pulled, code sets unrecorded:\n")
for concept, meta in sorted(summary.items()):
    dbs = ",".join(sorted(meta["databases"]))
    print(f"  {concept:24s} {meta['num_codes']:>4} codes   {dbs}   {meta['citation']}")


8 concepts pulled, code sets unrecorded:

  cardiometabolic            44 codes   D158,D76   NVSR 70-08 Table C p.10 (heart disease)
  chronic_liver               3 codes   D158,D76   NVSR 70-08 Table C p.10
  despair_composite         160 codes   D158,D76   union: alcohol_induced+drug_induced+suicide
  drug_induced              125 codes   D158,D76   NVSR 70-08 p.74
  drug_overdose              16 codes   D158,D76   NVSR 70-08 p.74 (subcategory)
  firearm                    14 codes   D158,D76   NVSR 70-08 p.75
  homicide                   28 codes   D158,D76   NVSR 70-08 p.11,40,43
  suicide                    27 codes   D158,D76   NVSR 70-08 Table C p.10
